In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, lag, datediff, sum, min, max
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Step 1: Sample data
data = [
    ("E001", "2024-06-01"),
    ("E001", "2024-06-02"),
    ("E001", "2024-06-03"),
    ("E001", "2024-06-05"),
    ("E002", "2024-06-01"),
    ("E002", "2024-06-03"),
    ("E002", "2024-06-04")
]

df = spark.createDataFrame(data, ["emp_id", "login_date"]) \
    .withColumn("login_date", to_date("login_date"))

# Step 2: Window spec
window_spec = Window.partitionBy("emp_id").orderBy("login_date")

# Step 3: Previous login date
df = df.withColumn("prev_date", lag("login_date").over(window_spec))

# Step 4: Gap between logins
df = df.withColumn("gap", datediff(col("login_date"), col("prev_date")))

display(df)

In [0]:
# Step 5: Mark start of new streak (gap != 1 or first record)
df = df.withColumn(
    "is_new_streak",
    ((col("gap") != 1) | col("gap").isNull()).cast("int")
)

display(df)

In [0]:
# Step 6: Assign streak id using cumulative sum
df = df.withColumn(
    "streak_id",
    sum("is_new_streak").over(window_spec.rowsBetween(Window.unboundedPreceding, 0))
)

# Step 7: Aggregate streak start, end, and days present
result = (
    df.groupBy("emp_id", "streak_id")
      .agg(
          min("login_date").alias("start_date"),
          max("login_date").alias("end_date"),
          (datediff(max("login_date"), min("login_date")) + 1).alias("days_present")
      )
      .orderBy("emp_id", "start_date")
)

result.show()
